# Step 1 V2 — Multi-Seed Safety Fine-Tuning & Subspace Extraction

**NeurIPS additions over V1:**
- Runs over `SEEDS = [42, 123, 2024, 7, 99]`
- Saves per-seed subspaces for downstream stability analysis
- Computes singular value spectrum per layer
- Plots subspace quality metrics
- Supports ablation over `safety_rank k ∈ {1,2,4,8,16,32}`

**Produces:**
- `models/safety_lora_seed{seed}/` — LoRA checkpoint per seed
- `models/safety_subspace_seed{seed}.pt` — dict with `bases` + `singular_values`
- `results/step1_subspace_quality.pt` — aggregated metrics
- `plots/singular_value_spectrum.png`
- `plots/subspace_rank_ablation.png`

In [ ]:
import sys, subprocess
print('=' * 60)
print('ENVIRONMENT')
print('=' * 60)
print(f'Python: {sys.version}')
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name} ({p.total_memory/1024**3:.1f} GB)')
print('=' * 60)

In [ ]:
%pip install -q transformers datasets peft evaluate matplotlib seaborn

In [ ]:
import sys
sys.path.insert(0, '.')   # so 'from utils import *' finds V2/utils.py
from utils import *

import matplotlib.pyplot as plt
import seaborn as sns
from transformers import TrainingArguments, Trainer, default_data_collator

# ── Step 1 specific config ────────────────────────────────────────────────────
RANK_ABLATION_VALUES = [1, 2, 4, 8, 16, 32]   # k values for ablation
MAIN_SAFETY_RANK = 8                            # used for main results

print(f'Seeds          : {SEEDS}')
print(f'Safety rank k  : {MAIN_SAFETY_RANK}')
print(f'Rank ablation  : {RANK_ABLATION_VALUES}')
print(f'Device         : {DEVICE}')
print(f'Models dir     : {MODELS_DIR}')

In [ ]:
def run_safety_finetune(model, tokenizer, chosen_texts, output_dir, seed):
    """Fine-tune on chosen (safe) responses. Returns trained model."""
    dataset = TextDataset(chosen_texts, tokenizer)
    args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=1,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR_FINETUNE,
        bf16=True,
        logging_steps=20,
        save_steps=500,
        save_total_limit=1,
        remove_unused_columns=False,
        report_to='none',
        dataloader_num_workers=0,
        seed=seed,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=default_data_collator,
    )
    trainer.train()
    model.save_pretrained(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    return model

print('run_safety_finetune defined.')

In [ ]:
# ── Main multi-seed loop ───────────────────────────────────────────────────────
all_singular_values = {}   # seed -> {layer_name: S tensor}
step1_metrics = []         # list of dicts, one per seed

for seed in SEEDS:
    lora_path = MODELS_DIR / f'safety_lora_seed{seed}'
    subspace_path = MODELS_DIR / f'safety_subspace_seed{seed}.pt'
    flag = MODELS_DIR / f'step1_seed{seed}_done.flag'

    print(f'\n{"="*55}')
    print(f'SEED {seed}')
    print(f'{"="*55}')

    if flag.exists():
        print(f'  Already done. Loading subspace...')
        ss = SafetySubspace.load(subspace_path, safety_rank=MAIN_SAFETY_RANK)
        all_singular_values[seed] = ss.singular_values
        step1_metrics.append({'seed': seed, 'n_layers': len(ss.bases)})
        continue

    # Load data for this seed
    chosen_texts, rejected_texts = load_hh_rlhf(
        n_chosen=N_SAFETY_SAMPLES, n_rejected=N_BIASED_SAMPLES, seed=seed
    )
    # Save rejected texts and DPO pairs for downstream steps
    torch.save(rejected_texts, str(DATA_DIR / f'rejected_texts_seed{seed}.pt'))
    dpo_pairs = list(zip(chosen_texts[:N_DPO_PAIRS], rejected_texts[:N_DPO_PAIRS]))
    torch.save(dpo_pairs, str(DATA_DIR / f'dpo_pairs_seed{seed}.pt'))

    # Load model and wrap with LoRA
    base_model, tokenizer = load_base_model_and_tokenizer(gpu_id=0)
    lora_model = apply_lora(base_model)

    # Safety fine-tune
    print(f'  Fine-tuning on {len(chosen_texts)} chosen responses...')
    lora_model = run_safety_finetune(lora_model, tokenizer, chosen_texts, lora_path, seed)

    # Extract safety subspace
    ss = SafetySubspace(safety_rank=MAIN_SAFETY_RANK)
    ss.compute_from_lora(lora_model)
    ss.save(subspace_path)

    all_singular_values[seed] = ss.singular_values
    step1_metrics.append({'seed': seed, 'n_layers': len(ss.bases)})
    print(f'  Subspace: {len(ss.bases)} layers, rank={MAIN_SAFETY_RANK}')

    flag.touch()
    del base_model, lora_model
    torch.cuda.empty_cache()

torch.save(step1_metrics, str(RESULTS_DIR / 'step1_metrics.pt'))
print('\nAll seeds complete:', step1_metrics)

In [ ]:
# ── Plot 1: Singular value spectrum ───────────────────────────────────────────
# For seed=42, plot the top-32 singular values for a sample of layers
ss_ref = SafetySubspace.load(MODELS_DIR / 'safety_subspace_seed42.pt')
layer_names = list(ss_ref.singular_values.keys())

# Sample 6 representative layers
nl = len(layer_names)
sample_layers = [layer_names[int(i * nl / 6)] for i in range(6)]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, lname in zip(axes.flat, sample_layers):
    S = ss_ref.singular_values[lname].float().numpy()
    ax.bar(range(1, len(S) + 1), S / S.sum(), color='steelblue', edgecolor='black', linewidth=0.4)
    ax.set_title(lname.split('.')[-2] + '.' + lname.split('.')[-1], fontsize=8)
    ax.set_xlabel('Singular value index', fontsize=7)
    ax.set_ylabel('Normalized energy', fontsize=7)
    ax.axvline(x=8, color='red', linestyle='--', linewidth=1, label=f'k=8')
    ax.legend(fontsize=7)

plt.suptitle('Safety Subspace: Singular Value Spectrum (seed=42)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'singular_value_spectrum.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: singular_value_spectrum.png')

In [ ]:
# ── Plot 2: Energy captured vs. rank k for seed=42 ────────────────────────────
ss_ref = SafetySubspace.load(MODELS_DIR / 'safety_subspace_seed42.pt')

mean_energy_at_k = []
for k in RANK_ABLATION_VALUES:
    energies = []
    for lname, S in ss_ref.singular_values.items():
        S = S.float()
        if len(S) >= k:
            captured = S[:k].pow(2).sum() / (S.pow(2).sum() + 1e-12)
            energies.append(captured.item())
    mean_energy_at_k.append(np.mean(energies))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(RANK_ABLATION_VALUES, mean_energy_at_k, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.axvline(x=8, color='red', linestyle='--', linewidth=1.5, label='k=8 (default)')
ax.set_xlabel('Safety rank k', fontsize=12)
ax.set_ylabel('Mean energy captured (all layers)', fontsize=12)
ax.set_title('Safety Subspace Energy vs. Rank k', fontsize=13, fontweight='bold')
ax.set_xticks(RANK_ABLATION_VALUES)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / 'subspace_rank_ablation_energy.png'), dpi=150, bbox_inches='tight')
plt.show()

for k, e in zip(RANK_ABLATION_VALUES, mean_energy_at_k):
    print(f'  k={k:2d}: {e:.4f} mean energy captured')

In [ ]:
# ── Plot 3: Subspace stability across seeds ────────────────────────────────────
# Measure cosine similarity between safety bases for seed=42 and other seeds
import itertools

loaded = {}
for seed in SEEDS:
    path = MODELS_DIR / f'safety_subspace_seed{seed}.pt'
    if path.exists():
        loaded[seed] = SafetySubspace.load(path)

if len(loaded) > 1:
    seed_pairs = list(itertools.combinations(list(loaded.keys()), 2))
    layer_names = list(loaded[SEEDS[0]].bases.keys())

    pair_similarities = []
    for s1, s2 in seed_pairs:
        sims = []
        for lname in layer_names:
            U1 = loaded[s1].bases.get(lname)
            U2 = loaded[s2].bases.get(lname)
            if U1 is None or U2 is None:
                continue
            # Subspace similarity via principal angles: ||U1^T U2||_F / k
            overlap = (U1.T @ U2).norm(p='fro').item() / U1.shape[1]
            sims.append(overlap)
        pair_similarities.append((f'{s1}-{s2}', np.mean(sims)))

    labels = [p[0] for p in pair_similarities]
    values = [p[1] for p in pair_similarities]

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.bar(labels, values, color='mediumseagreen', edgecolor='black', linewidth=0.7)
    ax.set_xlabel('Seed pair', fontsize=11)
    ax.set_ylabel('Mean subspace similarity (Frobenius overlap)', fontsize=11)
    ax.set_title('Safety Subspace Stability Across Seeds', fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.1)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', fontsize=9)
    ax.grid(axis='y', alpha=0.4)
    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / 'subspace_seed_stability.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('\nMean subspace similarity across all seed pairs:',
          f'{np.mean(values):.4f} ± {np.std(values):.4f}')
else:
    print('Need at least 2 seeds to compute stability.')

In [ ]:
print('Step 1 V2 complete.')
print(f'  Subspaces saved : {MODELS_DIR}')
print(f'  Plots saved     : {PLOTS_DIR}')
for m in step1_metrics:
    print(f'  seed={m["seed"]}: {m["n_layers"]} layers')